### Code configuration

In [1]:
!pip install tensorflow
!pip install keras-tuner
!pip install kaggle
!pip install typeguard>=4.4.2
!pip install transformers
!pip install tensorflow-addons
!pip install matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 10.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.4.1
    Uninstalling typeguard-4.4.1:
      Successfully uninstalled typeguard-4.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inflect 7.4.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.
ydata-profiling 4.12.2 requires typeguard<5,>=3, but you have typeguard 2.13.3 which is incompatible.


### Import necessary libraries :

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import (
    MobileNetV2, ResNet50,VGG16, VGG19, Xception, InceptionV3, EfficientNetB0,EfficientNetB7
)
from transformers import ViTModel,ViTFeatureExtractor,TFViTModel
from tensorflow.keras import layers, models
import os
import shutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from PIL import Image
import numpy as np
import keras_tuner as kt
from google.colab import files
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.applications import (
    xception, inception_v3, mobilenet_v2, vgg16, vgg19, efficientnet, resnet
)

### GPU check


In [3]:
print("TensorFlow version:", tf.__version__)
physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(physical_devices))
if physical_devices:
    print("GPU Details:", physical_devices[0])

TensorFlow version: 2.17.1
Num GPUs Available:  2
GPU Details: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


### Set GPU Memory Growth:

In [4]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth set for GPU")
    except RuntimeError as e:
        print("Error setting memory growth:", e)

Memory growth set for GPU


### Define constants :

#### Define image sizes

In [5]:
model_img_sizes = {
    "MobileNetV2": (224, 224),
    "ResNet50": (224, 224),
    "VGG19": (224, 224),
    "Xception": (299, 299),
    "InceptionV3": (299, 299),
    "EfficientNetB0": (224, 224),
    "EfficientNetB7": (600, 600),
    "VisionTransformer": (224, 224)
}
BATCH_SIZE = 32

#### Define preprocessing dictionary

In [6]:
preprocess_dict = {
    "Xception": xception.preprocess_input,
    "InceptionV3": inception_v3.preprocess_input,
    "MobileNetV2": mobilenet_v2.preprocess_input,
    "VGG16": vgg16.preprocess_input,
    "VGG19": vgg19.preprocess_input,
    "EfficientNetB7": efficientnet.preprocess_input,
    "EfficientNetB0": efficientnet.preprocess_input,
    "ResNet50": resnet.preprocess_input,
    "VisionTransformer": lambda x: x  # Preprocessing handled in model for ViT
}

#### Define paths

In [7]:
BASE_DIR = '/kaggle/input/handwritten-signature-verification/data/data'
SPLIT_BASE_DIR = '/kaggle/working/split_dataset'  # Path for split datasets
TRAIN_DIR = os.path.join(SPLIT_BASE_DIR, 'train')
VAL_DIR = os.path.join(SPLIT_BASE_DIR, 'val')
TEST_DIR = os.path.join(SPLIT_BASE_DIR, 'test')
# Create checkpoints directory
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('split_dataset', exist_ok=True)

### Function to split the dataset into train, validation, and test sets :

In [8]:
def split_dataset(base_dir, train_dir, val_dir, test_dir, train_ratio=0.7, val_ratio=0.15):

    test_ratio = 1 - train_ratio - val_ratio

    # Create directories for train, val, and test with class subfolders
    for split in ['train', 'val', 'test']:
        for class_name in ['real', 'forged']:
            os.makedirs(os.path.join(SPLIT_BASE_DIR, split, class_name), exist_ok=True)

    # Split data for each class
    for class_name in ['real', 'forged']:
        class_path = os.path.join(base_dir, class_name)
        images = []

        for root, dirs, files in os.walk(class_path):
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
                    images.append(os.path.join(root, file))

        print(f"Total images in {class_name}: {len(images)}")

        # First split: train+val vs test
        train_val_images, test_images = train_test_split(images, test_size=test_ratio, random_state=42)
        # Second split: train vs val
        train_images, val_images = train_test_split(train_val_images,
                                                    test_size=val_ratio/(train_ratio + val_ratio),
                                                    random_state=42)

        # Copy images to respective directories with unique filenames
        for img in train_images:
            # Create a unique filename by replacing directory separators with underscores
            relative_path = os.path.relpath(img, class_path)
            safe_name = relative_path.replace(os.sep, '_')
            shutil.copy(img, os.path.join(train_dir, class_name, safe_name))

        for img in val_images:
            relative_path = os.path.relpath(img, class_path)
            safe_name = relative_path.replace(os.sep, '_')
            shutil.copy(img, os.path.join(val_dir, class_name, safe_name))

        for img in test_images:
            relative_path = os.path.relpath(img, class_path)
            safe_name = relative_path.replace(os.sep, '_')
            shutil.copy(img, os.path.join(test_dir, class_name, safe_name))

    # Print split summary
    print("Dataset split completed:")
    for split, dir_path in [('Train', train_dir), ('Validation', val_dir), ('Test', test_dir)]:
        real_count = len(os.listdir(os.path.join(dir_path, 'real')))
        forged_count = len(os.listdir(os.path.join(dir_path, 'forged')))
        print(f"{split}: {real_count} real, {forged_count} forged images")

# Split the dataset
split_dataset(BASE_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR)

Total images in real: 3188
Total images in forged: 2984
Dataset split completed:
Train: 2230 real, 2088 forged images
Validation: 479 real, 448 forged images
Test: 479 real, 448 forged images


### Removing damaged images:

In [9]:
def check_images(directory):

    removed_files = 0
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.verify()  # Verifies if the file is a valid image
                img.close()   # Close the file to free resources
            except Exception as e:
                print(f"Error with file {file_path}: {e}")
                try:
                    os.remove(file_path)
                    print(f"Removed invalid file: {file_path}")
                    removed_files += 1
                except Exception as remove_error:
                    print(f"Failed to remove {file_path}: {remove_error}")
    print(f"Total files removed from {directory}: {removed_files}")

# Define directories
TRAIN_DIR = "/kaggle/working/split_dataset/train"
VAL_DIR = "/kaggle/working/split_dataset/val"
TEST_DIR = "/kaggle/working/split_dataset/test"

# Check each directory and remove invalid files
print("Checking training directory...")
check_images(TRAIN_DIR)
print("Checking validation directory...")
check_images(VAL_DIR)
print("Checking test directory...")
check_images(TEST_DIR)

Checking training directory...
Total files removed from /kaggle/working/split_dataset/train: 0
Checking validation directory...
Error with file /kaggle/working/split_dataset/val/forged/ru.d1c664d6-431d-4985-9e21-5989b1b7df4b__0001e12109--61ff34f8f73c4b173c32986d__JP A.jpg: cannot identify image file '/kaggle/working/split_dataset/val/forged/ru.d1c664d6-431d-4985-9e21-5989b1b7df4b__0001e12109--61ff34f8f73c4b173c32986d__JP A.jpg'
Removed invalid file: /kaggle/working/split_dataset/val/forged/ru.d1c664d6-431d-4985-9e21-5989b1b7df4b__0001e12109--61ff34f8f73c4b173c32986d__JP A.jpg
Total files removed from /kaggle/working/split_dataset/val: 1
Checking test directory...
Total files removed from /kaggle/working/split_dataset/test: 0


### Define data generators :

In [10]:
# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation and test generators (only rescaling)
val_test_datagen = ImageDataGenerator(rescale=1./255)


### Function to get data generators :

In [11]:
def get_datagens(model_name):
    """Return train and validation/test data generators based on model name."""
    if model_name == "VisionTransformer":
        # For ViT, rescale to [0, 1] and handle normalization in the model
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest'
        )
        val_test_datagen = ImageDataGenerator(rescale=1./255)
    else:
        # For other models, use the specific preprocess_input function
        preprocess_func = preprocess_dict[model_name]
        train_datagen = ImageDataGenerator(
            preprocessing_function=preprocess_func,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest'
        )
        val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_func)
    return train_datagen, val_test_datagen

### Define pre-trained models :

In [12]:
models_dict = {
     "Xception": Xception(weights='imagenet', include_top=False, input_shape=(299, 299, 3)),
     "InceptionV3": InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3)),
     "MobileNetV2": MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3)),
     "VGG16": VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3)),
     "VGG19": VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3)),
     "EfficientNetB7": EfficientNetB7(weights='imagenet', include_top=False, input_shape=(600, 600, 3)),
     "EfficientNetB0": EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3)),
     "VisionTransformer": TFViTModel.from_pretrained('google/vit-base-patch16-224', output_attentions=False),
     "ResNet50": ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3)),
}

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
258076736/258076736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFViTModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing TFViTModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFViTModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFViTModel were not initialized from the PyTorch model and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


### Function to build and train a model :

In [13]:
def build_model_with_hp(hp, base_model, name):
    img_size = model_img_sizes[name]
    batch_size = hp.Int('batch_size', min_value=16, max_value=64, step=16)

    # Model architecture with tunable parameters
    if name == "VisionTransformer":
        inputs = layers.Input(shape=(img_size[0], img_size[1], 3))
        # Inputs are in [0, 1] from data generator; apply ViT-specific normalization
        mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
        std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
        x = (inputs - mean) / std
        vit_outputs = base_model(x).last_hidden_state[:, 0, :]
        x = layers.Dense(
            units=hp.Int('dense_units', min_value=64, max_value=256, step=64),
            activation='relu'
        )(vit_outputs)
        x = layers.Dropout(hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1))(x)
        outputs = layers.Dense(1, activation='sigmoid')(x)
        model = tf.keras.Model(inputs, outputs)
    else:
        base_model.trainable = False
        model = models.Sequential([
            base_model,
            layers.GlobalAveragePooling2D(),
            layers.Dense(
                units=hp.Int('dense_units', min_value=64, max_value=256, step=64),
                activation='relu'
            ),
            layers.Dropout(hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1)),
            layers.Dense(1, activation='sigmoid')
        ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-2, sampling='log')
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

### Train and evaluate Function :

In [14]:
def train_and_evaluate_model(model, model_name, train_dir, val_dir, test_dir, batch_size=32, epochs=10, train_datagen=None, val_test_datagen=None):
    """Train and evaluate a model after hyperparameter tuning, with checkpoint saving."""
    img_size = model_img_sizes[model_name]

    # Define data generators
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary',
        shuffle=True
    )

    val_generator = val_test_datagen.flow_from_directory(
        val_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary',
        shuffle=True
    )

    test_generator = val_test_datagen.flow_from_directory(
        test_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary',
        shuffle=False
    )

    # Define checkpoint callback
    checkpoint_path = f"checkpoints/{model_name}_best.h5"
    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    )

    print(f"\nTraining {model_name} with tuned hyperparameters...")
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
            checkpoint_callback
        ],
        verbose=1
    )

    print(f"Evaluating {model_name} on test set...")
    test_loss, test_accuracy = model.evaluate(test_generator)

    # Generate predictions and evaluation metrics
    y_pred_probs = model.predict(test_generator)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()
    test_generator.reset()
    y_true = test_generator.classes

    f1 = f1_score(y_true, y_pred, average='weighted')
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    results = {
        'train_accuracy': history.history['accuracy'][-1],
        'val_accuracy': history.history['val_accuracy'][-1],
        'test_accuracy': test_accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'confusion_matrix': conf_matrix.tolist()
    }

    return results


### Train and evaluate all models :

In [ ]:
# Set up MirroredStrategy for multi-GPU training
strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")

results = {}
for model_name, base_model in models_dict.items():
    
    # Get model-specific data generators
    train_datagen, val_test_datagen = get_datagens(model_name)

    # Define the tuner
    tuner = kt.Hyperband(
        lambda hp: build_model_with_hp(hp, base_model, model_name),
        objective='val_accuracy',
        max_epochs=10,
        factor=3,
        directory='my_dir',
        project_name=f'tune_{model_name}',
        distribution_strategy = strategy
    )

    # Search for the best hyperparameters
    tuner.search(
        train_datagen.flow_from_directory(
            TRAIN_DIR,
            target_size=model_img_sizes[model_name],
            batch_size=32,
            class_mode='binary',
            shuffle=True
        ),
        validation_data=val_test_datagen.flow_from_directory(
            VAL_DIR,
            target_size=model_img_sizes[model_name],
            batch_size=32,
            class_mode='binary',
            shuffle=True
        ),
        epochs=10,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)]
    )

    # Get the best hyperparameters
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

    # Build the model with best hyperparameters
    with strategy.scope():
        best_model = build_model_with_hp(best_hps, base_model, model_name)

    # Train and evaluate the model separately
    results[model_name] = train_and_evaluate_model(
        best_model,
        model_name,
        TRAIN_DIR,
        VAL_DIR,
        TEST_DIR,
        batch_size=best_hps.get('batch_size'),
        train_datagen=train_datagen,
        val_test_datagen=val_test_datagen
    )

    # Clean up
    del best_model
    tf.keras.backend.clear_session()

Trial 14 Complete [00h 04m 38s]
val_accuracy: 0.6868250370025635

Best val_accuracy So Far: 0.6911447048187256
Total elapsed time: 01h 05m 42s

Search: Running Trial #15

Value             |Best Value So Far |Hyperparameter
64                |32                |batch_size
128               |256               |dense_units
0.4               |0.4               |dropout
6.2604e-05        |0.0017966         |learning_rate
4                 |4                 |tuner/epochs
2                 |2                 |tuner/initial_epoch
2                 |2                 |tuner/bracket
1                 |1                 |tuner/round
0006              |0003              |tuner/trial_id

Epoch 3/4
135/135 ━━━━━━━━━━━━━━━━━━━━ 137s 931ms/step - accuracy: 0.6488 - loss: 0.6267 - val_accuracy: 0.6350 - val_loss: 0.6329
Epoch 4/4
 80/135 ━━━━━━━━━━━━━━━━━━━━ 47s 855ms/step - accuracy: 0.6506 - loss: 0.6317

### Print final results :

In [ ]:
print("\nFinal Results:")
for model_name, metrics in results.items():
    print(f"\n{model_name.upper()}:")
    print("-" * (len(model_name) + 1))
    print(f"| Training Accuracy   | {metrics['train_accuracy']:.4f} |")
    print(f"| Validation Accuracy | {metrics['val_accuracy']:.4f} |")
    print(f"| Test Accuracy       | {metrics['test_accuracy']:.4f} |")
    print(f"| F1 Score            | {metrics['f1_score']:.4f} |")
    print(f"| Precision           | {metrics['precision']:.4f} |")
    print(f"| Recall              | {metrics['recall']:.4f} |")

    # Create a confusion matrix heatmap
    print("\nConfusion Matrix:")
    conf_matrix = np.array(metrics['confusion_matrix'])

    class_names = ['Fake', 'Real']

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True
    )
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.show()